# 05 — 保存済み古典6規模分析

3/4/5/25/99/125顧客の保存集計を比較します。処理は読み込みと列選択で、大規模最適化・SUMOは再実行しません。

**実行方法**：[統合ガイドの実行手順](../docs/QUANTITATIVE_GUIDE.md#10-quickstart-and-dependencies)の環境を用意し、上からRun All。作業ディレクトリはrepo rootまたはnotebooks。第三者の住所・大容量入力・SUMOは不要です。途中の条件を編集したら、そのセル以降を再実行してください。正式実験条件・freezeは変更しません。

**前提・範囲**：Python 3.12系で確認する小型教材／保存結果表示です。必要な配送を省略した電力量減を省エネ成果としません。量子実機・量子優位性・全需要充足・全システムエネルギーの実証ではありません。Colab実行は未検証です。


In [1]:
from pathlib import Path
import sys, json, math
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/adapters.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリrootまたはnotebooksを作業ディレクトリにしてください')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'notebooks'))
import adapters as api
S = '01_research_design/scenario_evaluation/'
M = '01_research_design/methodology/'
Q = '05_src/traffic_simulation/'
L = '05_src/literature_analysis/'


In [2]:
method_battery, scale_analysis = api.analysis_api()
routes=scale_analysis.routes()
assert routes['n'].tolist()==[3,4,5,25,99,125]
display(routes[['scale_id','n','planned_logical_instances','distinct_physical_instances','valid_logical_routes','logical_PROVED_OPTIMAL','logical_BOUNDED_GAP','travel_time_s_median','relative_gap_median']])
methods=scale_analysis.methods()
display(methods[['scale_id','n_customers','classical_solver','classical_guarantee','quantum_scope']])

,scale_id,n,planned_logical_instances,distinct_physical_instances,valid_logical_routes,logical_PROVED_OPTIMAL,logical_BOUNDED_GAP,travel_time_s_median,relative_gap_median
0,R0,3,78,78,78,78,0,1187.718786,0.000000
1,R1,4,78,78,78,78,0,1274.611313,0.000000
2,R2,5,78,78,78,78,0,1320.300955,0.000000
3,R3,25,78,78,78,0,78,2037.274146,0.049357
4,R4,99,78,78,78,0,78,3526.237451,0.119757
5,R5,125,78,70,78,0,78,3936.744754,0.143063


,scale_id,n_customers,classical_solver,classical_guarantee,quantum_scope
0,R0,3,EXHAUSTIVE_PERMUTATIONS,PROVED_OPTIMAL,FROZEN_CANDIDATE_C_PROTOCOL_V2
1,R1,4,EXHAUSTIVE_PERMUTATIONS,PROVED_OPTIMAL,FROZEN_CANDIDATE_C_PROTOCOL_V2
2,R2,5,EXHAUSTIVE_PERMUTATIONS,PROVED_OPTIMAL,NO_FINAL_COMMON_PROTOCOL_RESULT
3,R3,25,DETERMINISTIC_HEURISTIC_3_SWEEPS_PLUS_HIGHS_AS...,BOUNDED_GAP_NOT_GLOBAL_OPTIMUM,NO_FINAL_COMMON_PROTOCOL_RESULT
4,R4,99,DETERMINISTIC_HEURISTIC_3_SWEEPS_PLUS_HIGHS_AS...,BOUNDED_GAP_NOT_GLOBAL_OPTIMUM,NO_FINAL_COMMON_PROTOCOL_RESULT
5,R5,125,DETERMINISTIC_HEURISTIC_3_SWEEPS_PLUS_HIGHS_AS...,BOUNDED_GAP_NOT_GLOBAL_OPTIMUM,NO_FINAL_COMMON_PROTOCOL_RESULT


In [3]:
effects=scale_analysis.battery_summary()
rolling_case='CRR_LOWER_BOUND'
battery_effect='B2'  # B1/B2、表示だけ変更
selected=effects[(effects.rolling_case==rolling_case)&(effects.battery_effect==battery_effect)]
assert len(selected)==6
display(selected)
validation=scale_analysis.trajectory_validation()
print('保存診断で時間窓上限超過の停止:',validation['native_starts_after_due_diagnostic'])

,scale_id,rolling_case,battery_effect,count,planned_logical_count,distinct_physical_count,mean,min,P25,median,P75,P95,max,interpretation
1,R0,CRR_LOWER_BOUND,B2,78,78,78,2.256750,2.075974,2.236902,2.264896,2.289654,2.305670,2.330780,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...
5,R1,CRR_LOWER_BOUND,B2,78,78,78,2.260468,2.088104,2.241291,2.267599,2.291038,2.308221,2.332368,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...
9,R2,CRR_LOWER_BOUND,B2,78,78,78,2.266313,2.107659,2.244358,2.274174,2.299042,2.314057,2.335079,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...
13,R3,CRR_LOWER_BOUND,B2,78,78,78,2.321791,2.190037,2.302264,2.336400,2.348114,2.365805,2.380926,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...
17,R4,CRR_LOWER_BOUND,B2,78,78,78,2.380453,2.294345,2.368952,2.389174,2.400292,2.413751,2.428022,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...
21,R5,CRR_LOWER_BOUND,B2,78,78,70,2.390954,2.309494,2.379457,2.399721,2.409537,2.424029,2.430993,DESCRIPTIVE_LOGICAL_DISTRIBUTION_NOT_INDEPENDE...


保存診断で時間窓上限超過の停止: 136


## 読取り方と限界
travel_time_sは道路旅行時間の秒、Battery effectのmeanは削減率%です。logical件数とdistinct physical件数を区別し、同じ走行の論理重複を独立標本と扱いません。小規模は全列挙、大規模はヒューリスティック＋下界で、全規模に厳密最適性を主張しません。顧客構成・地理・時間窓・解法も変わるため、顧客数だけの因果効果ではありません。

保存された静的可行経路数は実時刻時間窓充足・実業務充足の保証ではありません。Batteryの段階比較は同一軌跡条件。全6規模は古典分析であり、量子Scalabilityの証拠ではありません。